# Geopack SDK: Data Management and Task Orchestration

This notebook demonstrates advanced data management features: uploading new files, exporting existing datasets to various formats, and monitoring background tasks.

---

### 🚀 Setup Note
This notebook is configured to work both with an installed `geopack-sdk` package or directly from the repository source code (`src/` folder).

---

In [1]:
# Enable auto-reload for development
%load_ext autoreload
%autoreload 2

import os
import sys
from dotenv import load_dotenv

# --- SMART SOURCE IMPORT ---
try:
    current_dir = os.getcwd()
    source_path = os.path.abspath(os.path.join(current_dir, "..", "src"))
    if os.path.exists(source_path):
        if source_path not in sys.path:
            sys.path.insert(0, source_path)
        print(f"ℹ️ Using SDK from local source: {source_path}")
    else:
        print("ℹ️ Using SDK from installed site-packages (pip)")
except Exception:
    print("⚠️ Could not determine local source path, falling back to pip.")

from geopack_sdk import (
    GeopackClient,
    GeopackAPIError,
    GeopackAuthError,
    GeopackError,
    GeopackTaskError,
    GeopackTimeoutError,
    task_log_entries_needing_review,
    task_message_badge_severity,
    task_message_info,
)

load_dotenv()
client = GeopackClient(base_url=os.getenv("GEOPACK_API_URL", "http://localhost:3000/api"))

try:
    client.auth.login(
        os.getenv("GEOPACK_USERNAME", "admin"),
        os.getenv("GEOPACK_PASSWORD", "password"),
    )
    print("✅ Login successful!")
except GeopackAuthError as e:
    print(f"❌ Authentication failed (HTTP {e.status_code}): {e.message}")
except GeopackAPIError as e:
    print(f"❌ API error during login (HTTP {e.status_code}): {e.message}")
except GeopackError as e:
    print(f"❌ Login failed: {e.message}")



ℹ️ Using SDK from local source: d:\Works\geopack-geoportal\geopack-geoportal-v2\python-sdk\src
✅ Login successful!


## Handling SDK Errors

The SDK raises typed exceptions (`GeopackAuthError`, `GeopackAPIError`, `GeopackTaskError`, `GeopackTimeoutError`) instead of generic `Exception`.


In [2]:
def report_sdk_error(context: str, exc: Exception) -> None:
    """Print a readable message for Geopack SDK exceptions."""
    if isinstance(exc, GeopackAuthError):
        print(f"[{context}] Auth error (HTTP {exc.status_code}): {exc.message}")
    elif isinstance(exc, GeopackAPIError):
        print(f"[{context}] API error (HTTP {exc.status_code}): {exc.message}")
    elif isinstance(exc, GeopackTaskError):
        print(f"[{context}] Task {exc.task_id} {exc.status}: {exc.message}")
    elif isinstance(exc, GeopackTimeoutError):
        print(f"[{context}] Timeout: {exc.message}")
    elif isinstance(exc, GeopackError):
        print(f"[{context}] {exc.message}")
    else:
        print(f"[{context}] Unexpected: {exc}")


try:
    client.datasets.get(999999999)
except GeopackAPIError as e:
    report_sdk_error("datasets.get (demo)", e)



[datasets.get (demo)] API error (HTTP 404): Dataset not found or access denied.


## 1. Environment Preparation
Before uploading, we need to choose a **DataStore** (where the data will be stored) and a **Workgroup** (who will own the data).

In [3]:
# 1. List available DataStores
datastores_response = client.datastores.list()
datastores = datastores_response.datastores
print("Available DataStores:")
for ds in datastores:
    ds_id = ds.id if isinstance(ds, dict) else ds.id
    ds_name = ds.name if isinstance(ds, dict) else ds.name
    ds_type = ds.get('type') if isinstance(ds, dict) else ds.type
    print(f"- [{ds_id}] {ds_name} ({ds_type})")


# 2. List available Workgroups
workgroups_response = client.workgroups.list()
workgroups = workgroups_response.workgroups
print("\nAvailable Workgroups:")
for wg in workgroups:
    wg_id = wg.id if isinstance(wg, dict) else wg.id
    wg_name = wg.name if isinstance(wg, dict) else wg.name
    print(f"- [{wg_id}] {wg_name}")

# --- SET YOUR TARGETS HERE ---
# By default, we'll use the first ones found if you don't change these
SELECTED_DATASTORE_ID = datastores[0].id if isinstance(datastores[0], dict) else datastores[0].id if datastores else 1
SELECTED_WORKGROUP_ID = workgroups[0].id if isinstance(workgroups[0], dict) else workgroups[0].id if workgroups else 1

print(f"\n✅ Ready to use: DataStore ID={SELECTED_DATASTORE_ID}, Workgroup ID={SELECTED_WORKGROUP_ID}")


Available DataStores:
- [11] Default Filesystem GDB (filesystem)
- [9] Default PostgreSQL GDB (postgres)
- [10] Default SQL Server GDB (mssql)
- [31] GDB_Golbahar (esri-geodatabase-mssql)
- [13] pg_test (postgres)
- [32] pg3 (postgres)
- [27] sql123 (mssql)

Available Workgroups:
- [1] Default Workgroup
- [4] gdb-kr-postgres
- [6] multi-criteria-overlay
- [5] Rigan bam
- [3] w2

✅ Ready to use: DataStore ID=11, Workgroup ID=1


## 1. Uploading a New Dataset
The SDK handles the two-step upload process: temporary file upload followed by background processing.

In [ ]:
import os

test_file = "data/Rural_District.shp.geojson"

if os.path.exists(test_file):
    print(f"Uploading {test_file}...")
    try:
        task_result = client.datasets.upload(
            file_path=test_file,
            data_store_id=SELECTED_DATASTORE_ID,
            workgroup_id=SELECTED_WORKGROUP_ID,
            wait=True,
        )
    except GeopackTaskError as e:
        report_sdk_error("upload task", e)
        raise
    except GeopackAPIError as e:
        report_sdk_error("upload API", e)
        raise
    except GeopackTimeoutError as e:
        report_sdk_error("upload wait", e)
        raise

    results = task_result.results or []
    print("\n✅ Upload successful! New datasets created:")
    if results:
        for ds in results:
            ds_id = getattr(ds, "createdDatasetId", None) or (
                ds.get("createdDatasetId") if isinstance(ds, dict) else None
            )
            ds_name = getattr(ds, "datasetName", None) or (
                ds.get("datasetName") if isinstance(ds, dict) else None
            )
            print(f"- {ds_name} (ID: {ds_id})")
    else:
        print("! Task finished but results field is empty.")
else:
    print(f"⚠️ File '{test_file}' not found. Put it under notebooks/data/.")



## 2. Exporting and Downloading Data
Need to download a dataset in a specific format? Geopack orchestrates an export task.

In [5]:
import os
# Let's export the first available dataset to GeoPackage
datasets = client.datasets.list(page_size=1)
if datasets.datasets:
    ds_id = datasets.datasets[0].id
    print(f"Exporting Dataset #{ds_id} to GPKG...")
    
    # 1. Start export and wait for completion
    task_result = client.datasets.export(dataset_id=ds_id, workgroup_id=1, format='gpkg', wait=True)
    
    # 2. Download the resulting file (convert TaskResult to dict for download)
    os.makedirs("downloads", exist_ok=True)
    task_result_dict = task_result.model_dump()
    local_file = client.datasets.download(task_result_dict, "downloads/")
    
    file_size = os.path.getsize(local_file) / (1024 * 1024)
    print(f"✅ Downloaded: {local_file} ({file_size:.2f} MB)")


Exporting Dataset #2406 to GPKG...
✅ Downloaded: d:\Works\geopack-geoportal\geopack-geoportal-v2\python-sdk\notebooks\downloads\Rural_District_shp.gpkg (2.42 MB)


## 3. Direct Task Monitoring
You can also monitor any background task manually using its ID.

In [6]:
# Example: Getting status of a specific task
if 'task_result' in locals():
    taskId = task_result.taskId
    status = client.tasks.get_status(taskId)
    print(f"Manual check for task {taskId}: {status.status}")


Manual check for task e163e3fe-1160-4816-9275-1c0be1be85b2: completed


### 3.1 Task log severity (portal parity)

After `get_status`, the SDK can mirror the Task History **Messages** column ([`TasksListView.vue` → `getMessageInfo`](../../frontend-ui/src/features/tasks/views/TasksListView.vue)): `task_message_info` (count, `has_errors`, `has_warnings`), `task_message_badge_severity` (`"error"` > `"warn"` > `"info"`), and `task_log_entries_needing_review` for warn/error lines. **`completed`** can still show a red/orange badge if the log contains error/warn entries.

In [ ]:
# Full task fetch (list rows may omit heavy fields) — then match portal message severity
if "task_result" in locals():
    tid = task_result.taskId
    t = client.tasks.get_status(tid)
    info = task_message_info(t)
    severity = task_message_badge_severity(t)
    print(
        f"Task {tid}: status={t.status}, messages={info.count}, "
        f"badge_severity={severity}"
    )

    if severity in ("error", "warn"):
        print(
            "Warning: this task has log lines flagged as error or warn "
            "(same highlight as the web UI)."
        )
        for line in task_log_entries_needing_review(t):
            level = line.get("level", "")
            msg = line.get("message", "")
            ts = line.get("timestamp", "")
            print(f"  [{level}] {ts} {msg}")

## 4. Task and API Error Handling

Uploads/exports use background tasks. Use `GeopackTaskError` and `GeopackTimeoutError` when calling `wait=True` or `tasks.wait_for_task()`.


In [ ]:
try:
    client.datasets.get(999999999)
except GeopackAPIError as e:
    report_sdk_error("invalid dataset id", e)

try:
    client.tasks.wait_for_task(
        "00000000-0000-0000-0000-000000000000",
        timeout=5,
        interval=1,
        quiet=True,
    )
except GeopackAPIError as e:
    report_sdk_error("unknown task id", e)
except GeopackTaskError as e:
    report_sdk_error("task failed", e)
except GeopackTimeoutError as e:
    report_sdk_error("task wait", e)

